# 01 — Collect MovieLens 25M + IMDb TSVs (Step 1)

**What:** Downloads MovieLens 25M and the IMDb non-commercial TSVs, verifies each, and records metadata.

**Why:** MovieLens is the CF backbone (`links.csv` gives movieId↔imdbId↔tmdbId for Step 2's join). IMDb adds external ratings/vote counts and crew, joined later via `imdbId`/`tconst`. TMDB/Wikipedia/Reddit are deferred to after Step 4's quality tiers.

**Inputs:** none — pulls directly from `files.grouplens.org` and `datasets.imdbws.com`.

**Outputs:**
- `data/raw/movielens/ml-25m.zip`, `ml-25m/*.csv`, `metadata.json`
- `data/raw/imdb/<snapshot_date>/*.tsv.gz`, `metadata.json`

Safe to re-run: every stage checks for its own saved output first and skips the expensive work if it's already there and verified.

## Config
All paths, URLs, and the published checksum/row counts/schema to validate against.

In [1]:
import gzip
import hashlib
import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# --- MovieLens ---
ML25M_URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
ML25M_MD5 = "6b51fb2759a8657d3bfcbfc42b592ada"  # published at files.grouplens.org/datasets/movielens/ml-25m-README.html

ML_RAW_DIR = Path("..") / "data" / "raw" / "movielens"
ML_ZIP_PATH = ML_RAW_DIR / "ml-25m.zip"
ML_EXTRACT_DIR = ML_RAW_DIR / "ml-25m"
ML_METADATA_PATH = ML_RAW_DIR / "metadata.json"

ML_EXPECTED_FILES = [
    "ratings.csv", "movies.csv", "links.csv",
    "tags.csv", "genome-scores.csv", "genome-tags.csv",
]
ML_EXPECTED_ROW_COUNTS = {
    "ratings.csv": 25_000_095,
    "movies.csv": 62_423,
    "tags.csv": 1_093_360,
}

# --- IMDb ---
IMDB_BASE_URL = "https://datasets.imdbws.com"
IMDB_FILES = ["title.basics.tsv.gz", "title.ratings.tsv.gz", "title.crew.tsv.gz"]
IMDB_EXPECTED_COLUMNS = {
    "title.basics.tsv.gz": {
        "tconst", "titleType", "primaryTitle", "originalTitle", "isAdult",
        "startYear", "endYear", "runtimeMinutes", "genres",
    },
    "title.ratings.tsv.gz": {"tconst", "averageRating", "numVotes"},
    "title.crew.tsv.gz": {"tconst", "directors", "writers"},
}
IMDB_SNAPSHOT_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
IMDB_RAW_DIR = Path("..") / "data" / "raw" / "imdb" / IMDB_SNAPSHOT_DATE
IMDB_METADATA_PATH = IMDB_RAW_DIR / "metadata.json"

ML_RAW_DIR.mkdir(parents=True, exist_ok=True)
IMDB_RAW_DIR.mkdir(parents=True, exist_ok=True)

## Helper functions
Kept small and self-contained (no notebook-global state beyond the config above) so they can move into `src/` unchanged at Step 4.

In [2]:
def md5sum(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def download(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                f.write(chunk)


def read_gzip_header(path: Path) -> list[str]:
    with gzip.open(path, "rt") as f:
        return f.readline().strip().split("\t")

## MovieLens — Stage: Download
Pulls the ~250MB archive only if it isn't already sitting in `data/raw/`.

In [3]:
if ML_ZIP_PATH.exists():
    print(f"{ML_ZIP_PATH} already exists, skipping download (will verify checksum next).")
else:
    print(f"Downloading {ML25M_URL} -> {ML_ZIP_PATH}")
    download(ML25M_URL, ML_ZIP_PATH)
    print("Download complete.")

../data/raw/movielens/ml-25m.zip already exists, skipping download (will verify checksum next).


## MovieLens — Stage: Verify
Confirms the archive matches the checksum MovieLens publishes.

In [4]:
ml_actual_md5 = md5sum(ML_ZIP_PATH)
if ml_actual_md5 != ML25M_MD5:
    raise ValueError(f"MD5 mismatch: expected {ML25M_MD5}, got {ml_actual_md5}")
print("Checksum verified.")

Checksum verified.


## MovieLens — Stage: Extract
Unzips into `data/raw/movielens/ml-25m/`, skipping if that folder already exists.

In [5]:
if ML_EXTRACT_DIR.exists():
    print(f"{ML_EXTRACT_DIR} already exists, skipping extraction.")
else:
    print(f"Extracting to {ML_EXTRACT_DIR}")
    with zipfile.ZipFile(ML_ZIP_PATH) as zf:
        zf.extractall(ML_RAW_DIR)
    print("Extraction complete.")

../data/raw/movielens/ml-25m already exists, skipping extraction.


## MovieLens — Stage: Validate
Loads each CSV once, checks row counts against MovieLens's documented figures, and spot-checks a known mapping (movieId 1 = Toy Story = tt0114709). Skips the re-read if `metadata.json` already confirms this passed.

In [6]:
if ML_METADATA_PATH.exists():
    print(f"{ML_METADATA_PATH} already exists — loading instead of re-reading all CSVs.")
    ml_row_counts = json.loads(ML_METADATA_PATH.read_text())["row_counts"]
    ml_spot_check_passed = True
else:
    ml_row_counts = {}
    for fname in ML_EXPECTED_FILES:
        fpath = ML_EXTRACT_DIR / fname
        if not fpath.exists():
            raise FileNotFoundError(f"Expected file missing after extraction: {fpath}")
        df = pd.read_csv(fpath)
        ml_row_counts[fname] = len(df)
        print(f"  {fname}: {len(df):,} rows, columns={list(df.columns)}")

    for fname, expected in ML_EXPECTED_ROW_COUNTS.items():
        if ml_row_counts[fname] != expected:
            raise ValueError(
                f"{fname}: expected {expected:,} rows per MovieLens docs, got {ml_row_counts[fname]:,}"
            )

    links = pd.read_csv(ML_EXTRACT_DIR / "links.csv")
    toy_story_imdb = links.loc[links["movieId"] == 1, "imdbId"].iloc[0]
    if int(toy_story_imdb) != 114709:
        raise ValueError(f"Spot check failed: movieId 1 -> imdbId {toy_story_imdb}, expected 114709")
    ml_spot_check_passed = True
    print("Spot check passed: movieId 1 -> tt0114709 (Toy Story)")

../data/raw/movielens/metadata.json already exists — loading instead of re-reading all CSVs.


## MovieLens — Stage: Save
Writes the metadata file that the Validate stage checks for on re-run.

In [7]:
if not ML_METADATA_PATH.exists():
    ml_metadata = {
        "source_url": ML25M_URL,
        "md5": ML25M_MD5,
        "downloaded_at": datetime.now(timezone.utc).isoformat(),
        "row_counts": ml_row_counts,
        "spot_check": "movieId 1 -> imdbId 114709 (Toy Story) verified",
    }
    ML_METADATA_PATH.write_text(json.dumps(ml_metadata, indent=2))
    print(f"Metadata written to {ML_METADATA_PATH}")
else:
    print(f"{ML_METADATA_PATH} already exists, not overwriting.")

../data/raw/movielens/metadata.json already exists, not overwriting.


## IMDb — Stage: Download
Pulls all three TSVs for today's snapshot date, skipping entirely if a verified `metadata.json` for today already exists. No separate extract stage — the TSVs are read directly from gzip, no unzip needed. IMDb republishes daily with no stable checksum, so each run is stamped with today's date rather than deduplicated by content hash.

In [8]:
imdb_already_done = IMDB_METADATA_PATH.exists()
if imdb_already_done:
    print(f"{IMDB_METADATA_PATH} already exists for {IMDB_SNAPSHOT_DATE}, skipping download.")
else:
    for fname in IMDB_FILES:
        dest = IMDB_RAW_DIR / fname
        if dest.exists():
            print(f"{dest} already present, skipping.")
            continue
        print(f"Downloading {IMDB_BASE_URL}/{fname} -> {dest}")
        download(f"{IMDB_BASE_URL}/{fname}", dest)

../data/raw/imdb/2026-10-05/metadata.json already exists for 2026-10-05, skipping download.


## IMDb — Stage: Verify
Checks each file's header against the expected IMDb schema.

In [9]:
if not imdb_already_done:
    for fname in IMDB_FILES:
        header = read_gzip_header(IMDB_RAW_DIR / fname)
        missing = IMDB_EXPECTED_COLUMNS[fname] - set(header)
        if missing:
            raise ValueError(f"{fname}: missing expected columns {missing}")
        print(f"{fname}: schema OK")
else:
    print("Skipped (already validated this run).")

Skipped (already validated this run).


## IMDb — Stage: Validate
Loads each TSV, records row counts, and spot-checks tt0114709 -> Toy Story in `title.basics`.

In [10]:
if imdb_already_done:
    imdb_row_counts = json.loads(IMDB_METADATA_PATH.read_text())["row_counts"]
    imdb_spot_check_passed = True
    print("Loaded row counts from existing metadata.json instead of re-reading TSVs.")
else:
    imdb_row_counts = {}
    for fname in IMDB_FILES:
        df = pd.read_csv(IMDB_RAW_DIR / fname, sep="\t", compression="gzip", na_values="\\N", low_memory=False)
        imdb_row_counts[fname] = len(df)
        print(f"  {fname}: {len(df):,} rows, columns={list(df.columns)}")

    basics = pd.read_csv(
        IMDB_RAW_DIR / "title.basics.tsv.gz", sep="\t", compression="gzip", na_values="\\N", low_memory=False
    )
    toy_story = basics.loc[basics["tconst"] == "tt0114709", "primaryTitle"]
    if toy_story.empty or toy_story.iloc[0] != "Toy Story":
        raise ValueError(f"Spot check failed: tt0114709 lookup returned {toy_story.tolist()}")
    imdb_spot_check_passed = True
    print("Spot check passed: tt0114709 -> Toy Story")

Loaded row counts from existing metadata.json instead of re-reading TSVs.


## IMDb — Stage: Save
Writes the metadata file that the Download/Validate stages check for on re-run.

In [11]:
if not imdb_already_done:
    imdb_metadata = {
        "source_url": IMDB_BASE_URL,
        "snapshot_date": IMDB_SNAPSHOT_DATE,
        "downloaded_at": datetime.now(timezone.utc).isoformat(),
        "row_counts": imdb_row_counts,
        "spot_check": "tt0114709 -> Toy Story verified",
    }
    IMDB_METADATA_PATH.write_text(json.dumps(imdb_metadata, indent=2))
    print(f"Metadata written to {IMDB_METADATA_PATH}")
else:
    print(f"{IMDB_METADATA_PATH} already exists, not overwriting.")

../data/raw/imdb/2026-10-05/metadata.json already exists, not overwriting.


## Summary

In [12]:
print("Data collection summary (Step 1)")
print("=" * 40)
print("MovieLens 25M:")
print(f"  Archive: {ML_ZIP_PATH} ({'OK' if ML_ZIP_PATH.exists() else 'MISSING'})")
print(f"  Extracted: {ML_EXTRACT_DIR} ({'OK' if ML_EXTRACT_DIR.exists() else 'MISSING'})")
print(f"  Checksum verified: {ml_actual_md5 == ML25M_MD5}")
print(f"  Spot check passed: {ml_spot_check_passed}")
for fname, count in ml_row_counts.items():
    print(f"    {fname}: {count:,}")

print("\nIMDb TSVs:")
print(f"  Snapshot date: {IMDB_SNAPSHOT_DATE}")
print(f"  Metadata: {IMDB_METADATA_PATH} ({'OK' if IMDB_METADATA_PATH.exists() else 'MISSING'})")
print(f"  Spot check passed: {imdb_spot_check_passed}")
for fname, count in imdb_row_counts.items():
    print(f"    {fname}: {count:,}")

Data collection summary (Step 1)
MovieLens 25M:
  Archive: ../data/raw/movielens/ml-25m.zip (OK)
  Extracted: ../data/raw/movielens/ml-25m (OK)
  Checksum verified: True
  Spot check passed: True
    ratings.csv: 25,000,095
    movies.csv: 62,423
    links.csv: 62,423
    tags.csv: 1,093,360
    genome-scores.csv: 15,584,448
    genome-tags.csv: 1,128

IMDb TSVs:
  Snapshot date: 2026-10-05
  Metadata: ../data/raw/imdb/2026-10-05/metadata.json (OK)
  Spot check passed: True
    title.basics.tsv.gz: 12,838,880
    title.ratings.tsv.gz: 1,717,073
    title.crew.tsv.gz: 12,837,873
